## 1. Imports

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F

## 2. Configuration
Set the input file path, output table name, and the target chunk size (10 MB). Also read basic file info (name, size, modified date) and define the table schema.

`LARGE_ARRAY_FIELD` is the key in the JSON whose array holds the bulk of the data — this is the field that gets split into chunks.

In [0]:
file_path = "/Volumes/iics/casemix/history/json_test/sample_500_mb.json"

target_table = "iics.casemix.json_test_500_mb_variant_chunks"

TARGET_SIZE = 10 * 1024 * 1024

file_info = dbutils.fs.ls(file_path)[0]

file_name = file_info.name

file_size = file_info.size

file_modified = datetime.fromtimestamp(
    file_info.modificationTime / 1000
).date()

LARGE_ARRAY_FIELD = "contained"

schema = """
    file_name STRING,
    file_path STRING,
    file_size_bytes BIGINT,
    file_modified DATE,
    file_stucture STRING
"""

## 3. Load JSON and split into chunks
Read the full JSON file into memory, pull out the large array, and keep everything else as `metadata`.

Loop through each entry, adding it to the current chunk until the chunk reaches ~10 MB, then start a new chunk. Each chunk is repackaged as its own JSON document (metadata + a slice of the array) and stored as a row.

In [0]:
with open(file_path, "rb") as f:

    data = json.load(f)


if LARGE_ARRAY_FIELD not in data:

    raise Exception(
        f"Array field '{LARGE_ARRAY_FIELD}' was not found."
    )


entries = data[LARGE_ARRAY_FIELD]

metadata = {
    key: value
    for key, value in data.items()
    if key != LARGE_ARRAY_FIELD
}


print("Number of objects:", len(entries))



rows = []

chunk = []
chunk_size = 0
chunk_no = 0


for entry in entries:

    entry_json = json.dumps(
        entry,
        separators=(",", ":"),
        ensure_ascii=False
    )

    entry_size = len(
        entry_json.encode("utf-8")
    )

    
    if (chunk and chunk_size + entry_size > TARGET_SIZE):

        chunk_no += 1

        chunk_document = metadata.copy()

        chunk_document[LARGE_ARRAY_FIELD] = chunk

        chunk_json = json.dumps(
            chunk_document,
            separators=(",", ":"),
            ensure_ascii=False
        )

        rows.append(
            (
                file_name,
                file_path,
                file_size,
                file_modified,
                chunk_no,
                chunk_json
            )
        )

        chunk = []
        chunk_size = 0




    chunk.append(entry)

    chunk_size += entry_size


if chunk:

    chunk_no += 1

    chunk_document = metadata.copy()

    chunk_document[LARGE_ARRAY_FIELD] = chunk

    chunk_json = json.dumps(
        chunk_document,
        separators=(",", ":"),
        ensure_ascii=False
    )

    rows.append(
        (
            file_name,
            file_path,
            file_size,
            file_modified,
            chunk_no,
            chunk_json
        )
    )


total_chunks = len(rows)

print("Total chunks:", total_chunks)

## 4. Create Spark DataFrame
Convert the list of chunk rows into a Spark DataFrame using the schema defined earlier.

In [0]:
df = spark.createDataFrame(
    rows,
    schema=schema
)

## 5. Add chunk size column
Compute the byte size of each chunk's JSON text and add it as a new column.

In [0]:
df = df.withColumn(
    "chunk_size_bytes",
    F.length(
        F.encode(
            F.col("json_text"),
            "UTF-8"
        )
    )
)

## 6. Parse JSON as VARIANT
Select the metadata columns plus the total chunk count, and parse `json_text` into Spark's `VARIANT` type so it's queryable as structured JSON.

In [0]:
variant_df = (
    df
    .select(
        "file_name",
        "file_path",
        "file_size_bytes",
        "file_modified",
        "chunk_no",

        F.lit(total_chunks)
        .cast("int")
        .alias("total_chunks"),

        "chunk_size_bytes",

        F.parse_json(
            F.col("json_text")
        ).alias("json_data")
    )
)

In [0]:
variant_df = df.withColumn("loaded_at", F.current_timestamp())

## 7. Save to Delta table
Write the final DataFrame to the target Delta table, overwriting any existing data.

In [0]:
(
    variant_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

## Testing

In [0]:
%sql
SELECT chunk_no,
       SUBSTRING(CAST(json_data AS STRING), 1, 50000)      AS part_1,
       SUBSTRING(CAST(json_data AS STRING), 50001, 50000)  AS part_2,
       SUBSTRING(CAST(json_data AS STRING), 100001, 50000) AS part_3
FROM iics.casemix.json_test_500_mb_variant_chunks
WHERE file_name = 'sample_500_mb.json'
  AND chunk_no = 3;


In [0]:
%sql
SELECT 
 file_name
, file_path
, file_size_bytes
, file_modified
, chunk_no
, total_chunks
, chunk_size_bytes
-- , json_data
FROM iics.casemix.json_test_500_mb_variant_chunks

In [0]:
%sql
DESCRIBE iics.casemix.json_test_500_mb_variant_chunks